**MQTT Topics:**

| Topic | Direction | Payload |
|-------|-----------|---------|
| `team08/hospital/control` | Frontend sends | `{"command": "select_drone", "drone": "drone-01"}` |
| `team08/hospital/control` | Frontend sends | `{"command": "medicine_loaded", "drone": "drone-01"}` |
| `team08/hospital/control` | Frontend sends | `{"command": "return", "drone": "drone-01"}` |
| `team08/hospital/control` | Frontend sends | `{"command": "back"}` |
| `team08/hospital/display` | Backend publishes | `{"status": "selected_drone", "drone": "drone-01", "message": "..."}` |
| `team08/hospital/display` | Backend publishes | `{"status": "medicine_loaded", "drone": "drone-01", "message": "..."}` |
| `team08/hospital/display` | Backend publishes | `{"status": "returning", "drone": "drone-01", "message": "..."}` |
| `team08/hospital/display` | Backend publishes | `{"status": "initiated", "message": "..."}` |

In [1]:
import json
from threading import Thread

import paho.mqtt.client as mqtt
from stmpy import Driver, Machine

In [ ]:
broker, port = "mqtt20.iik.ntnu.no", 1883

MQTT_TOPIC_CONTROL = "team08/hospital/control"
MQTT_TOPIC_DISPLAY = "team08/hospital/display"


class HospitalFrontend:
    
    def display_drone(self, drone):
        message = "Drone {} selected.".format(drone)
        print(message)
        payload = json.dumps({"status": "selected_drone", "drone": drone, "message": message})
        self.mqtt_client.publish(MQTT_TOPIC_DISPLAY, payload)

    def display_overview(self):
        message = "Back to overview. Showing all drones."
        print(message)
        payload = json.dumps({"status": "overview_all_drones", "message": message})
        self.mqtt_client.publish(MQTT_TOPIC_DISPLAY, payload)

    def transit_to_drone_return(self, drone):
        message = "Drone {} is returning.".format(drone)
        print(message)
        payload = json.dumps({"status": "returning", "drone": drone, "message": message})
        self.mqtt_client.publish(MQTT_TOPIC_CONTROL, payload)

    def transmit_medicine_loaded(self, drone):
        message = "Medicine loaded on drone {}.".format(drone)
        print(message)
        payload = json.dumps({"status": "medicine_loaded", "drone": drone, "message": message})
        self.mqtt_client.publish(MQTT_TOPIC_CONTROL, payload)
        
    def on_init(self):
        print("Hospital frontend initialized. Showing overview of all drones.")
        message = "System is initiated."
        payload = json.dumps({"status": "initiated", "message": message})
        self.mqtt_client.publish(MQTT_TOPIC_DISPLAY, payload)

In [3]:
drone_frontend = HospitalFrontend()

# Init
t0 = {
    "source": "initial",
    "target": "overview_all_drones",
    "effect": "on_init",
}

# Bruker velger en drone
t1 = {
    "trigger": "select_drone",
    "source": "overview_all_drones",
    "target": "selected_drone",
    "effect": "display_drone(*)",
}

# Bruker går tilbake til oversikt
t2 = {
    "trigger": "back",
    "source": "selected_drone",
    "target": "overview_all_drones",
    "effect": "display_overview",
}

# Drone returnerer
t3 = {
    "trigger": "return",
    "source": "selected_drone",
    "target": "selected_drone",
    "effect": "transit_to_drone_return(*)",
}

# Medisin lastet på drone
t4 = {
    "trigger": "medicine_loaded",
    "source": "selected_drone",
    "target": "selected_drone",
    "effect": "transmit_medicine_loaded(*)",
}

drone_machine = Machine(
    name="hospital_frontend",
    transitions=[t0, t1, t2, t3, t4],
    obj=drone_frontend,
)
drone_frontend.stm = drone_machine


class HospitalMQTTClient:

    def __init__(self):
        self.client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
        self.client.on_connect = self.on_connect
        self.client.on_message = self.on_message

    def on_connect(self, client, userdata, flags, reason_code, properties):
        print("on_connect(): {}".format(reason_code))

    def on_message(self, client, userdata, msg):
        print("on_message(): topic: {} payload: {}".format(msg.topic, msg.payload.decode("utf-8")))

        try:
            payload = json.loads(msg.payload.decode("utf-8"))
        except json.JSONDecodeError:
            print("Invalid JSON payload, ignoring message.")
            return

        if msg.topic == MQTT_TOPIC_BUZZ:
            drone = payload.get("drone", "Unknown")
            print("Drone selected: {}".format(drone))
            self.stm_driver.send("select_drone", "hospital_frontend", args=[drone])

        elif msg.topic == MQTT_TOPIC_CONTROL:
            command = payload.get("command", "")
            if command == "return":
                drone = payload.get("drone", "Unknown")
                print("Drone returning: {}".format(drone))
                self.stm_driver.send("return", "hospital_frontend", args=[drone])
            elif command == "medicine_loaded":
                drone = payload.get("drone", "Unknown")
                print("Medicine loaded on drone: {}".format(drone))
                self.stm_driver.send("medicine_loaded", "hospital_frontend", args=[drone])
            elif command == "back":
                print("Back to overview")
                self.stm_driver.send("back", "hospital_frontend")
            else:
                print("Unknown command: {}".format(command))
        
        elif msg.topic == MQTT_TOPIC_CONTROL:
            command = payload.get("command", "")
            drone = payload.get("drone", "Unknown")

            if command == "select_drone":
                self.stm_driver.send("select_drone", "hospital_frontend", args=[drone])
            elif command == "medicine_loaded":
                self.stm_driver.send("medicine_loaded", "hospital_frontend", args=[drone])
            elif command == "return":
                self.stm_driver.send("return", "hospital_frontend", args=[drone])
            elif command == "back":
                self.stm_driver.send("back", "hospital_frontend")

    def start(self, broker, port):
        print("Connecting to {}:{}".format(broker, port))
        self.client.connect(broker, port)
        self.client.subscribe(MQTT_TOPIC_BUZZ)
        self.client.subscribe(MQTT_TOPIC_CONTROL)
        try:
            thread = Thread(target=self.client.loop_forever)
            thread.start()
        except KeyboardInterrupt:
            print("Interrupted")
            self.client.disconnect()

In [ ]:
driver = Driver()
driver.add_machine(drone_machine)

myclient = HospitalMQTTClient()
drone_frontend.mqtt_client = myclient.client
myclient.stm_driver = driver

driver.start(keep_active=True)
myclient.start(broker, port)
drone_frontend.on_init()

Hospital frontend initialized. Showing overview of all drones.Connecting to mqtt20.iik.ntnu.no:1883

Hospital frontend initialized. Showing overview of all drones.


on_connect(): Success


on_message(): topic: team08/hospital/control payload: {"command": "select_drone", "drone": "drone-01"}
Unknown command: select_drone


Machine hospital_frontend is in state overview_all_drones and received event medicine_loaded, but no transition with this event is declared!


on_message(): topic: team08/hospital/control payload: {"command": "medicine_loaded", "drone": "drone-01"}
Medicine loaded on drone: drone-01


...

on_message(): topic: team08/hospital/control payload: {"command": "return", "drone": "drone-01"}
Drone returning: drone-01


Machine hospital_frontend is in state overview_all_drones and received event return, but no transition with this event is declared!
Machine hospital_frontend is in state overview_all_drones and received event return, but no transition with this event is declared!


on_message(): topic: team08/hospital/control payload: {"command": "return", "drone": "drone-01"}
Drone returning: drone-01
on_message(): topic: team08/hospital/control payload: {"command": "return", "drone": "drone-01"}
Drone returning: drone-01


Machine hospital_frontend is in state overview_all_drones and received event return, but no transition with this event is declared!
Machine hospital_frontend is in state overview_all_drones and received event return, but no transition with this event is declared!


on_message(): topic: team08/hospital/control payload: {"command": "return", "drone": "drone-01"}
Drone returning: drone-01
on_message(): topic: team08/hospital/control payload: {"command": "return", "drone": "drone-01"}
Drone returning: drone-01
